In [ ]:
import requests
import json
from datetime import datetime, timedelta
import time

def download_binance_data(symbol="ETCUSDT", interval="5m"):
    """Download 3 years of data and save as JSON"""
    
    # 3 years ago to now
    end_time = int(datetime.now().timestamp() * 1000)
    start_time = int((datetime.now() - timedelta(days=3*365)).timestamp() * 1000)
    
    print(f"Downloading {symbol} {interval} data...")
    
    all_data = []
    current_start = start_time
    
    while current_start < end_time:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': current_start,
            'limit': 1000
        }
        
        response = requests.get("https://api.binance.com/api/v3/klines", params=params)
        data = response.json()
        
        if not data:
            break
            
        # Convert to clean format
        for item in data:
            all_data.append({
                'timestamp': item[0],
                'date': datetime.fromtimestamp(item[0]/1000).strftime('%Y-%m-%d %H:%M:%S'),
                'open': float(item[1]),
                'high': float(item[2]),
                'low': float(item[3]),
                'close': float(item[4]),
                'volume': float(item[5])
            })
        
        current_start = data[-1][0] + 1
        print(f"Downloaded {len(all_data)} records...")
        time.sleep(0.1)
    
    # Save to JSON (append if file exists)
    filename = f"{symbol}_{interval}_3years.json"
    
    # Load existing data if file exists
    existing_data = []
    try:
        with open(filename, 'r') as f:
            existing_data = json.load(f)
        print(f"Found existing file with {len(existing_data)} records")
    except FileNotFoundError:
        print("Creating new file")
    
    # Append new data to existing data
    existing_data.extend(all_data)
    
    # Remove duplicates based on timestamp and sort
    seen_timestamps = set()
    unique_data = []
    for item in existing_data:
        if item['timestamp'] not in seen_timestamps:
            unique_data.append(item)
            seen_timestamps.add(item['timestamp'])
    
    # Sort by timestamp (oldest first)
    unique_data.sort(key=lambda x: x['timestamp'])
    
    # Save updated data
    with open(filename, 'w') as f:
        json.dump(unique_data, f, indent=2)
    
    print(f"Saved {len(unique_data)} total records to {filename}")
    return unique_data

# Usage
if __name__ == "__main__":
    # Download Bitcoin 5-minute data
    download_binance_data("ETCUSDT", "5m")
